# 01 - File Tools Introduction

`FileTools` 是一個 `@dataclass`，將檔案系統操作封裝成三個工具函式：
- `list_files` — 列出目錄下的檔案
- `grep` — 跨檔案正則搜尋
- `read_file` — 讀取單一檔案（含路徑穿越防護）

這三個函式在後續筆記本中會被 LLM Agent（LangGraph ReAct）直接呼叫，作為 agent 的「工具層」。

In [9]:
import sys
from pathlib import Path

# Works whether Jupyter is started from project root or examples/
_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / "examples"
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from utils.tools import FileTools

## FileTools 簡介

`FileTools` 是一個以 `@dataclass` 實作的工具集，接受一個 `root` 路徑作為根目錄，所有操作均被限制在此根目錄之內。
內建路徑穿越防護（path traversal protection），任何試圖存取根目錄以外路徑的操作都會拋出 `ValueError`。

類別簽名概念如下：

```python
@dataclass
class FileTools:
    root: Path

    def list_files(self, pattern: str = "*.md") -> list[str]: ...
    def grep(self, pattern: str, max_results: int = 30) -> list[str]: ...
    def read_file(self, path: str) -> str: ...
```

In [10]:
crm = FileTools(PROJECT_ROOT / "data" / "crm_notes")
print(f"Root: {crm.root}")

Root: /home/mistin/research/agentic-rag-lab/data/crm_notes


## Tool 1 — list_files

`list_files(pattern="*.md")` 對根目錄執行 glob 搜尋，回傳相對於根目錄的路徑清單，並依字母排序。預設 pattern 為 `*.md`，可傳入任意 glob pattern 進行過濾。

In [11]:
files = crm.list_files()
for f in files:
    print(f)

meeting_001_晨星科技_2026-05-08.md
meeting_002_鴻圖零售_2026-05-15.md
meeting_003_晨星科技_2026-05-22.md
meeting_004_新星金融科技_2026-05-28.md


In [12]:
# Can also filter by pattern
crm.list_files("*晨星*")

['meeting_001_晨星科技_2026-05-08.md', 'meeting_003_晨星科技_2026-05-22.md']

## Tool 2 — grep

`grep(pattern, max_results=30)` 對根目錄下所有 `.md` 檔案執行正則表達式搜尋，回傳格式為 `"file:lineno: text"` 的字串清單。`max_results` 參數限制最大回傳筆數，避免結果過多佔用 LLM context window。

In [13]:
hits = crm.grep("UAT")
for h in hits:
    print(h)

meeting_001_晨星科技_2026-05-08.md:63: | R-004 | 客戶 IT 團隊學習曲線，影響 UAT 進度 | 低 | 高 | 提供上線前培訓計畫（3 場） |
meeting_001_晨星科技_2026-05-08.md:89: [資料遷移沙盒測試] ──► [UAT] ──► [上線準備] ──► [Go-Live]
meeting_001_晨星科技_2026-05-08.md:104: | UAT 執行 | 張志偉 + 許淑芬（客戶） | 2026-06-20 | UAT 簽核表 |
meeting_002_鴻圖零售_2026-05-15.md:88: [整合測試（SIT）] ──► [UAT] ──► [上線準備] ──► [Go-Live]
meeting_002_鴻圖零售_2026-05-15.md:103: | UAT 執行 | 黃建國 + 吳宗翰（客戶） | 2026-07-10 | UAT 簽核表 |
meeting_003_晨星科技_2026-05-22.md:64: | R-004 | 客戶 UAT 學習曲線（維持） | 低 | 高 | 維持培訓計畫 |
meeting_003_晨星科技_2026-05-22.md:95: [資料遷移驗證（縮減為 1 次）] ──► [UAT（壓縮為 3 天）] ──► [Go-Live]
meeting_003_晨星科技_2026-05-22.md:109: | UAT 執行（壓縮） | 張志偉 + 許淑芬（客戶） | 2026-06-13 | UAT 簽核表 | 從 5 天壓縮為 3 天 |
meeting_003_晨星科技_2026-05-22.md:121: - [ ] **[林佳慧]** 通知客戶方：許淑芬需預留 2026-06-11 至 06-13 進行 UAT — `Due: 2026-05-23`
meeting_004_新星金融科技_2026-05-28.md:97: [系統整合測試 SIT] ──► [UAT + 法遵驗證] ──► [驗收 Go-Live]
meeting_004_新星金融科技_2026-05-28.md:115: | UAT + 法遵驗證 | 鄭博文 + 劉承翰（客戶） | 2026-10-28 | UAT 簽核表 + 法遵確認函 |


In [14]:
# 跨檔案搜尋人名
crm.grep("陳建宏")

['meeting_001_晨星科技_2026-05-08.md:26: | 陳建宏 | 技術負責人（Tech Lead） | 技術評估 |',
 'meeting_001_晨星科技_2026-05-08.md:76: | D-004 | 技術規格書由我方（陳建宏）撰寫，客戶方（張志偉）確認 | 王雅婷 | 2026-05-08 |',
 'meeting_001_晨星科技_2026-05-08.md:85: 林佳慧           陳建宏              張志偉          陳建宏',
 'meeting_001_晨星科技_2026-05-08.md:91: 陳建宏           張志偉        王雅婷          王雅婷',
 'meeting_001_晨星科技_2026-05-08.md:100: | 技術規格書撰寫 | 陳建宏 | 2026-05-20 | 技術設計文件 |',
 'meeting_001_晨星科技_2026-05-08.md:102: | 雲端環境建置 | 陳建宏 | 2026-06-01 | 環境建置報告 |',
 'meeting_001_晨星科技_2026-05-08.md:103: | 資料遷移沙盒驗證（共 3 次） | 陳建宏 | 2026-06-10 | 遷移測試報告 |',
 'meeting_001_晨星科技_2026-05-08.md:112: - [ ] **[陳建宏]** 撰寫技術規格書，涵蓋 AWS 架構圖與資料流 — `Due: 2026-05-20`',
 'meeting_001_晨星科技_2026-05-08.md:115: - [ ] **[陳建宏]** 向玉山、國泰銀行索取最新 Open API 文件 — `Due: 2026-05-15`',
 'meeting_003_晨星科技_2026-05-22.md:26: | 陳建宏 | 技術負責人（Tech Lead） | 技術方案說明 |',
 'meeting_003_晨星科技_2026-05-22.md:38: 本次為 2026-05-08 初次會議之後續確認會議。我方已提交技術規格書 v1.0（陳建宏，2026-05-19 完成，提前一天），本次主要目的為技術簽核，但會中客戶資安長蔡國峰提出重大異議。',
 'm

## Tool 3 — read_file

`read_file(path)` 以相對路徑讀取根目錄下的單一檔案，回傳檔案全文字串。內建路徑穿越防護：若解析後的絕對路徑超出 `root` 範圍，則立即拋出 `ValueError`，不會執行讀取。

In [15]:
content = crm.read_file("meeting_001_晨星科技_2026-05-08.md")
print(content[:800])

# 客戶會議紀錄

## 基本資訊

| 欄位 | 內容 |
|------|------|
| **客戶名稱** | 晨星科技股份有限公司 |
| **會議日期** | 2026-05-08 |
| **會議時間** | 14:00 – 16:00 |
| **會議地點** | 晨星科技總部 3F 會議室 A |
| **紀錄人** | 王雅婷 |

## 參與者

### 客戶方
| 姓名 | 職稱 |
|------|------|
| 李明哲 | 資訊長（CIO） |
| 張志偉 | IT 部門主管 |
| 許淑芬 | 財務系統分析師 |

### 我方
| 姓名 | 職稱 | 角色 |
|------|------|------|
| 王雅婷 | 專案經理（PM） | 會議主持、紀錄 |
| 陳建宏 | 技術負責人（Tech Lead） | 技術評估 |
| 林佳慧 | 業務分析師 | 需求訪談 |

---

## 會議主題

ERP 系統升級 — 初步需求討論與可行性評估

---

## 客戶需求

1. 將現有地端 ERP（SAP ECC 6.0）升級至雲端版本，支援多廠區統一管理。
2. 財務模組需與現有銀行 API（玉山、國泰）串接，支援自動對帳。
3. 新系統需支援角色型存取控制（RBAC），並通過 ISO 27001 稽核要求。
4. 介面需提供中文化報表匯出（Excel、PDF），並相容現行 BI 工具（Power BI）。
5. 期望在 **2026-07-01** 正式上線（第一階段：財務模組）。

---

## 客戶痛點

- **痛點 1｜資料孤島**：各廠區資料各自維護，彙整報表需 2–3 天人工作業。
- **痛點 2｜稽核追蹤困難**：現有系統缺乏完整 Audit Log，法遵壓力大。
- **痛點 3｜舊系統效能瓶頸**：月結日


In [16]:
try:
    crm.read_file("../../../etc/passwd")
except ValueError as e:
    print(f"[Blocked] {e}")

[Blocked] Path '../../../etc/passwd' is outside the allowed directory


## 小結

以上三個工具（`list_files`、`grep`、`read_file`）共同構成了 Agentic RAG 系統的**檔案系統層（file system layer）**：

- `list_files` — 讓 agent 探索有哪些資料可用
- `grep` — 讓 agent 快速定位關鍵詞所在位置
- `read_file` — 讓 agent 取得完整文件內容進行推理

下一個筆記本（`02_crm_tool_calling_agent.ipynb`）將展示如何讓 LangGraph ReAct Agent 自主、迭代地呼叫這三個工具，回答跨文件的複雜問題。